In [1]:
# Berechne die Total Variation Distance für die vier Methoden anhand der Datensätze
import numpy as np
import pandas as pd

from efficient_probit_regression.total_variation_distance import total_variation_distance
from efficient_probit_regression.sampling import compute_leverage_scores, calculate_lewis_weights_exact, calculate_l2_lp_leverage_score, compute_random_evaluations_probabilities_v2, to_density

print("Total Variation Distance")

# load all datasets

%store -r Webspam
%store -r Iris
%store -r KDDCup
%store -r Covertype
%store -r Example2D

p = [1.0, 1.3, 1.5, 1.7, 2.0]

def tvd_table_for_dataset(
    X: np.ndarray,
    dataset_name: str,
    p_values,
    T_lewis: int = 20,
    m_random: int = 1000,
    rng=2026,
) -> pd.DataFrame:
    """
    Erzeugt einen DataFrame mit allen 6 paarweisen TVDs der 4 Methoden
    (lev, lewis, l2_lp, random) für einen Datensatz und alle p-Werte.
    """

    rows = []

    for p in p_values:
        lev = compute_leverage_scores(X, p=p)
        lw  = calculate_lewis_weights_exact(X, p=p, T=T_lewis)
        l2lp = calculate_l2_lp_leverage_score(X, p=p)[1]
        rnd = compute_random_evaluations_probabilities_v2(X, m=m_random, p=p, rng=rng)

        # Dichten
        lev_d = to_density(lev)
        lw_d  = to_density(lw)

        # 6 Paar-Vergleiche
        tvd_lev_lewis   = round(total_variation_distance(lev, lw, normalize=True), 4)
        tvd_lev_l2_lp   = round(total_variation_distance(lev_d, l2lp), 4)
        tvd_lev_random  = round(total_variation_distance(lev_d, rnd), 4)
        tvd_l2_lp_lewis = round(total_variation_distance(l2lp, lw_d), 4)
        tvd_lewis_rand  = round(total_variation_distance(lw_d, rnd), 4)
        tvd_l2_lp_rand  = round(total_variation_distance(l2lp, rnd), 4)

        rows.append({
            "dataset": dataset_name,
            "p": float(p),
            "lev - lewis": float(tvd_lev_lewis),
            "lev - l2lp": float(tvd_lev_l2_lp),
            "lev - random": float(tvd_lev_random),
            "l2lp - lewis": float(tvd_l2_lp_lewis),
            "lewis - random": float(tvd_lewis_rand),
            "l2lp - random": float(tvd_l2_lp_rand),
        })

    df = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
    return df

datasets = {
    "Iris": Iris.X,
    "Example2D": Example2D.X,
    "Webspam": Webspam.X,
    "KDDCup": KDDCup.X,
    "Covertype": Covertype.X,
}

tvd_dfs = {}

for name, X in datasets.items():
    df = tvd_table_for_dataset(
        X,
        dataset_name=name,
        p_values=p,
        T_lewis=20,
        m_random=1000,
        rng=2026
    )
    tvd_dfs[name] = df

    print(f"\n=== {name} ===")
    display(df)

Total Variation Distance

=== Iris ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Iris,1.0,0.1323,0.1374,0.2088,0.0894,0.0987,0.1560
1,Iris,1.3,0.1138,0.1258,0.0982,0.0686,0.1205,0.1609
2,Iris,1.5,0.1634,0.1683,0.1137,0.0501,0.1334,0.1576
3,Iris,1.7,0.1633,0.1678,0.1856,0.0260,0.1452,0.1600
4,Iris,2.0,0.0000,0.0000,0.1614,0.0000,0.1614,0.1614



=== Example2D ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Example2D,1.0,0.1945,0.1971,0.1553,0.0993,0.0899,0.1207
1,Example2D,1.3,0.2219,0.2307,0.2526,0.0742,0.1173,0.1145
2,Example2D,1.5,0.1242,0.1589,0.2135,0.0549,0.1361,0.1366
3,Example2D,1.7,0.1991,0.1933,0.2899,0.0316,0.1548,0.1470
4,Example2D,2.0,0.0000,0.0000,0.1825,0.0000,0.1825,0.1825



=== Webspam ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Webspam,1.0,0.1137,0.1537,0.2910,0.1138,0.2664,0.3254
1,Webspam,1.3,0.1097,0.1699,0.2509,0.1062,0.2713,0.3286
2,Webspam,1.5,0.1386,0.1757,0.3232,0.0867,0.2734,0.3228
3,Webspam,1.7,0.3485,0.3435,0.5104,0.0514,0.2745,0.3009
4,Webspam,2.0,0.0000,0.0000,0.3017,0.0000,0.3017,0.3017



=== KDDCup ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,KDDCup,1.0,0.1673,0.1782,0.5807,0.1293,0.5475,0.5860
1,KDDCup,1.3,0.1551,0.2006,0.4181,0.0955,0.5003,0.5312
2,KDDCup,1.5,0.1843,0.1987,0.5016,0.0648,0.4479,0.4704
3,KDDCup,1.7,0.2861,0.2862,0.4924,0.0393,0.3736,0.3876
4,KDDCup,2.0,0.0000,0.0000,0.2945,0.0000,0.2945,0.2945



=== Covertype ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Covertype,1.0,0.2241,0.1914,0.1969,0.0771,0.3780,0.3406
1,Covertype,1.3,0.1928,0.2042,0.3076,0.0493,0.3846,0.3970
2,Covertype,1.5,0.2479,0.2534,0.3452,0.0424,0.3917,0.4043
3,Covertype,1.7,0.2707,0.2735,0.3867,0.0346,0.3976,0.4052
4,Covertype,2.0,0.0000,0.0000,0.4086,0.0000,0.4086,0.4086


In [3]:
import pandas as pd
from pathlib import Path

df_all = pd.concat(tvd_dfs.values(), ignore_index=True)

print("\n=== ALL DATASETS (combined) ===")
display(df_all)

out_dir = Path("Tabellen")
out_dir.mkdir(exist_ok=True, parents=True)
out_csv = "tvd_all_methods.csv"
df_all.to_csv(out_dir / out_csv, index=False, float_format="%.10f")
print("Saved:", out_csv)


=== ALL DATASETS (combined) ===


,dataset,p,lev - lewis,lev - l2lp,lev - random,l2lp - lewis,lewis - random,l2lp - random
0,Iris,1.0,0.1323,0.1374,0.2088,0.0894,0.0987,0.1560
1,Iris,1.3,0.1138,0.1258,0.0982,0.0686,0.1205,0.1609
2,Iris,1.5,0.1634,0.1683,0.1137,0.0501,0.1334,0.1576
3,Iris,1.7,0.1633,0.1678,0.1856,0.0260,0.1452,0.1600
4,Iris,2.0,0.0000,0.0000,0.1614,0.0000,0.1614,0.1614
5,Example2D,1.0,0.1945,0.1971,0.1553,0.0993,0.0899,0.1207
6,Example2D,1.3,0.2219,0.2307,0.2526,0.0742,0.1173,0.1145
7,Example2D,1.5,0.1242,0.1589,0.2135,0.0549,0.1361,0.1366
8,Example2D,1.7,0.1991,0.1933,0.2899,0.0316,0.1548,0.1470
9,Example2D,2.0,0.0000,0.0000,0.1825,0.0000,0.1825,0.1825


Saved: tvd_all_methods.csv
